# Scaling-sweep experiment — variational quantum meta-learning diagnostics

**Companion notebook for** *"Adaptation-trajectory diagnostics fail to predict observable generalization in variational quantum meta-learning"* (target: Physical Review A).

## What this tests
The manuscript's falsifiable physical prediction (Sec. IV C / Discussion):

> As variational circuits are scaled toward the barren-plateau regime, the concentration of circuit gradients increases (the coefficient of variation of the gradient norm shrinks), so the first-order linearization identity
> $$I_{\mathrm{val}} \approx \alpha\,\langle g_{\mathrm{val}}, g_{\mathrm{tr}}\rangle = \alpha\,\cos(g_{\mathrm{val}},g_{\mathrm{tr}})\,\lVert g_{\mathrm{val}}\rVert\,\lVert g_{\mathrm{tr}}\rVert$$
> grips the gradient-alignment diagnostic ever more tightly. Consequently gradient alignment becomes **more** strongly tied to *local* validation improvement $I_{\mathrm{val}}$ while remaining a **weak, seed-unstable** predictor of the *held-out observable gap* $G = L_{\mathrm{val}}(\theta_K) - L_{\mathrm{tr}}(\theta_K)$.

## Two bodies of evidence, both vs. $(n, L)$
**(A) Standard barren-plateau probe** (random inits): $\mathrm{Var}[\partial L_{\mathrm{tr}}/\partial\theta_1]$ and mean $\lVert g\rVert$ — the canonical McClean *et al.* signature.

**(B) The meta-learning diagnostic experiment** (the paper's setting): at the Reptile meta-init $\theta_0$, per task record $\lVert g_{\mathrm{tr}}\rVert,\lVert g_{\mathrm{val}}\rVert$, alignment $A=\cos(g_{\mathrm{tr}},g_{\mathrm{val}})$, then adapt $K$ steps and record $I_{\mathrm{val}}$ and $G$. Aggregate per $(n,L,\mathrm{seed})$.

## Prediction confirmed if
- mean $\lVert g\rVert$, $\mathrm{Var}[\partial L/\partial\theta_1]$, and $\mathrm{CV}(\lVert g\rVert)$ **decrease** with $n$ (and $L$) — gradient concentration;
- $\rho(A, I_{\mathrm{val}})$ stays **high** and `identity_gap` $\to 0$ — identity dominance;
- $\rho(A, G)$ stays **weak** and flips sign/magnitude **across seeds** — not a gap predictor.

> **Reproduce your exact task family:** if you have the v17 task generator, replace `draw_coeffs` and `split_indices` with your originals so absolute numbers match Tables I–II. The *scaling trends* (the object of the prediction) are robust to the distribution choice.

## 0. Imports

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import pennylane as qml
from pennylane import numpy as pnp

import matplotlib.pyplot as plt

print("pennylane", qml.__version__)

## 1. Task family — random anisotropic Heisenberg/XYZ with local $Z$ fields
$H_\tau=\sum_j c_{\tau,j}P_{\tau,j}$ on an $n$-qubit open chain. Pauli terms: nearest-neighbour $XX,YY,ZZ$ on each bond plus a local $Z$ on each site, so $M = 3(n-1)+n = 4n-3$ (\(n=6\Rightarrow21\), matching the paper). The operators are **fixed** for a given $n$; only the coefficients vary per task.

In [ ]:
def build_ops(n):
    """Fixed Pauli operators for an n-qubit open XYZ chain + local Z fields.
    Order: [XX_i, YY_i, ZZ_i for i=0..n-2] then [Z_i for i=0..n-1].  M = 4n-3.
    """
    ops = []
    for i in range(n - 1):
        ops.append(qml.PauliX(i) @ qml.PauliX(i + 1))
        ops.append(qml.PauliY(i) @ qml.PauliY(i + 1))
        ops.append(qml.PauliZ(i) @ qml.PauliZ(i + 1))
    for i in range(n):
        ops.append(qml.PauliZ(i))
    return ops


def draw_coeffs(rng, n, coupling_lo=0.5, coupling_hi=1.5, field_scale=0.5):
    """One task's coefficients: anisotropic couplings + random local Z fields.
    REPLACE with your v17 generator for exact parity with Tables I-II.
    """
    M = 4 * n - 3
    n_coupl = 3 * (n - 1)
    c = np.empty(M)
    signs = rng.choice([-1.0, 1.0], size=n_coupl)
    c[:n_coupl] = rng.uniform(coupling_lo, coupling_hi, size=n_coupl) * signs
    c[n_coupl:] = rng.normal(0.0, field_scale, size=n)
    return c


def split_indices(rng, M, support_frac=2.0 / 3.0):
    """Disjoint observable support/query split (fixed once per task)."""
    m_tr = max(1, min(M - 1, int(round(support_frac * M))))
    perm = rng.permutation(M)
    return perm[:m_tr], perm[m_tr:]

## 2. Ansatz and bounded subset loss
Hardware-efficient ansatz: $L$ layers of $(R_Y,R_Z)$ per qubit followed by a linear CNOT chain ($2nL$ parameters). The subset loss $\langle H_S\rangle/\sum_{j\in S}|c_j|\in[-1,1]$ is differentiated by backprop. (CV of the gradient norm is scale-invariant, so it is unaffected by this normalization.)

In [ ]:
def ansatz(theta, n, L):
    """theta shape (L, n, 2): L layers of (RY,RZ) per qubit + linear CNOT chain."""
    for l in range(L):
        for w in range(n):
            qml.RY(theta[l, w, 0], wires=w)
            qml.RZ(theta[l, w, 1], wires=w)
        for w in range(n - 1):
            qml.CNOT(wires=[w, w + 1])


def make_loss(dev, n, L, coeffs_subset, ops_subset):
    """Bounded subset loss <H_S>/sum|c|, differentiable via backprop."""
    denom = float(np.sum(np.abs(coeffs_subset))) or 1.0
    norm_coeffs = [float(c) / denom for c in coeffs_subset]
    H = qml.Hamiltonian(norm_coeffs, list(ops_subset))

    @qml.qnode(dev, diff_method="backprop")
    def qn(theta):
        ansatz(theta, n, L)
        return qml.expval(H)
    return qn


def flat_grad(qn, theta):
    """Flattened gradient of a scalar qnode at theta."""
    return np.asarray(qml.grad(qn)(theta), dtype=float).reshape(-1)

## 3. Reptile meta-initialization and inner adaptation
$\theta_0$ is produced by a Reptile outer loop; each evaluation task is then adapted for $K$ inner gradient steps on its **support** loss.

In [ ]:
def reptile_meta_init(dev, n, L, ops, rng, n_outer, K, inner_lr, outer_lr,
                      init_scale, coeff_kw):
    """Return a Reptile-style meta-initialization theta_0 (pnp array)."""
    theta = pnp.array(rng.normal(0.0, init_scale, size=(L, n, 2)),
                      requires_grad=True)
    M = len(ops)
    for _ in range(n_outer):
        c = draw_coeffs(rng, n, **coeff_kw)
        tr_idx, _ = split_indices(rng, M)
        qn_tr = make_loss(dev, n, L, c[tr_idx], [ops[j] for j in tr_idx])
        phi = pnp.array(theta, requires_grad=True)
        for _ in range(K):
            g = qml.grad(qn_tr)(phi)
            phi = pnp.array(phi - inner_lr * g, requires_grad=True)
        theta = pnp.array(theta + outer_lr * (phi - theta), requires_grad=True)
    return theta


def adapt(dev, n, L, theta0, ops, tr_idx, c, K, inner_lr):
    """K-step inner adaptation on the support loss; return adapted params."""
    qn_tr = make_loss(dev, n, L, c[tr_idx], [ops[j] for j in tr_idx])
    phi = pnp.array(theta0, requires_grad=True)
    for _ in range(K):
        g = qml.grad(qn_tr)(phi)
        phi = pnp.array(phi - inner_lr * g, requires_grad=True)
    return phi

## 4. (A) Barren-plateau probe at random initializations
Canonical signature: $\mathrm{Var}[\partial L_{\mathrm{tr}}/\partial\theta_1]$ should decay (≈ exponentially) with $n$ for expressive hardware-efficient ansätze.

In [ ]:
def barren_probe(dev, n, L, ops, rng, R, coeff_kw):
    """Var and mean of a single gradient component + norm stats at random init."""
    M = len(ops)
    comp, norms = [], []
    for _ in range(R):
        c = draw_coeffs(rng, n, **coeff_kw)
        tr_idx, _ = split_indices(rng, M)
        qn_tr = make_loss(dev, n, L, c[tr_idx], [ops[j] for j in tr_idx])
        th = pnp.array(rng.uniform(0.0, 2 * np.pi, size=(L, n, 2)),
                       requires_grad=True)
        g = flat_grad(qn_tr, th)
        comp.append(g[0]); norms.append(np.linalg.norm(g))
    comp = np.asarray(comp); norms = np.asarray(norms)
    mean_norm = float(np.mean(norms))
    return {
        "grad_comp_var": float(np.var(comp)),
        "grad_comp_absmean": float(np.mean(np.abs(comp))),
        "norm_mean": mean_norm,
        "norm_cv": float(np.std(norms) / mean_norm) if mean_norm > 0 else np.nan,
        "R": R,
    }

## 5. (B) Meta-learning diagnostic experiment
Per task, at $\theta_0$: $\lVert g_{\mathrm{tr}}\rVert,\lVert g_{\mathrm{val}}\rVert$, $A=\cos(g_{\mathrm{tr}},g_{\mathrm{val}})$, and $IP=\langle g_{\mathrm{tr}},g_{\mathrm{val}}\rangle$ (the identity predictor). After $K$-step adaptation: $I_{\mathrm{tr}}, I_{\mathrm{val}}, G$, drift.

In [ ]:
def barren_row(dev, n, L, ops, rng, R, coeff_kw, seed):
    d = barren_probe(dev, n, L, ops, rng, R, coeff_kw)
    d.update(dict(n=n, L=L, seed=seed, params=2 * n * L, M=len(ops)))
    return d


def meta_eval(dev, n, L, ops, theta0, rng, n_eval, K, inner_lr, coeff_kw):
    """Per-task diagnostics at theta_0 and after adaptation -> DataFrame.
    Records I_val at K=1 (where the one-step identity is exact) and at K."""
    M = len(ops)
    rows = []
    for t in range(n_eval):
        c = draw_coeffs(rng, n, **coeff_kw)
        tr_idx, val_idx = split_indices(rng, M)
        qn_tr = make_loss(dev, n, L, c[tr_idx], [ops[j] for j in tr_idx])
        qn_val = make_loss(dev, n, L, c[val_idx], [ops[j] for j in val_idx])

        g_tr = flat_grad(qn_tr, theta0)
        g_val = flat_grad(qn_val, theta0)
        ntr = float(np.linalg.norm(g_tr)); nval = float(np.linalg.norm(g_val))
        dot = float(np.dot(g_tr, g_val))
        A = dot / (ntr * nval) if (ntr > 0 and nval > 0) else np.nan
        IP = dot                                   # identity predictor = <g_tr,g_val>
        gtr1 = float(g_tr[0])                       # meta-init gradient component (for BP-at-theta0)

        Ltr0 = float(qn_tr(theta0)); Lval0 = float(qn_val(theta0))
        # one inner step -> K=1 improvement (identity is exact here)
        phi1 = adapt(dev, n, L, theta0, ops, tr_idx, c, 1, inner_lr)
        I_val_K1 = Lval0 - float(qn_val(phi1))
        # full K steps
        phi = adapt(dev, n, L, theta0, ops, tr_idx, c, K, inner_lr)
        LtrK = float(qn_tr(phi)); LvalK = float(qn_val(phi))

        I_tr = Ltr0 - LtrK; I_val = Lval0 - LvalK; G = LvalK - LtrK
        drift = float(np.linalg.norm(
            np.asarray(phi, dtype=float).reshape(-1)
            - np.asarray(theta0, dtype=float).reshape(-1)))
        rows.append(dict(task=t, norm_gtr=ntr, norm_gval=nval, align=A, IP=IP,
                         gtr1=gtr1, Ltr0=Ltr0, LtrK=LtrK, Lval0=Lval0, LvalK=LvalK,
                         I_tr=I_tr, I_val=I_val, I_val_K1=I_val_K1, G=G, drift=drift))
    return pd.DataFrame(rows)


def safe_spearman(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3 or np.std(x[m]) == 0 or np.std(y[m]) == 0:
        return np.nan, np.nan
    r, p = spearmanr(x[m], y[m]); return float(r), float(p)


def partial_spearman(x, y, z):
    """Partial Spearman rho of (x, y) controlling z.

    Method matches the manuscript: rank-transform all variables, regress the
    ranked x and ranked y on the ranked control z (linear), then correlate the
    residuals. This removes the part of the x-y association induced by the
    shared control z -- here rho(A, G | I_val): does alignment retain a gap
    association after the I_val-induced component is removed?
    """
    from scipy.stats import rankdata
    x = np.asarray(x, float); y = np.asarray(y, float); z = np.asarray(z, float)
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    if m.sum() < 4:
        return np.nan
    rx, ry, rz = rankdata(x[m]), rankdata(y[m]), rankdata(z[m])
    if np.std(rx) == 0 or np.std(ry) == 0 or np.std(rz) == 0:
        return np.nan
    def resid(a, b):
        s, i = np.polyfit(b, a, 1)
        return a - (s * b + i)
    ex, ey = resid(rx, rz), resid(ry, rz)
    if np.std(ex) == 0 or np.std(ey) == 0:
        return np.nan
    return float(np.corrcoef(ex, ey)[0, 1])


def summarize(df, n, L, seed):
    cv = lambda a: float(np.std(a) / np.mean(a)) if np.mean(a) > 0 else np.nan
    rA_Ival, pA_Ival = safe_spearman(df["align"], df["I_val"])
    rIP_Ival, _ = safe_spearman(df["IP"], df["I_val"])
    rA_G, pA_G = safe_spearman(df["align"], df["G"])
    rA_G_part = partial_spearman(df["align"], df["G"], df["I_val"])  # control I_val
    rIval_G, _ = safe_spearman(df["I_val"], df["G"])
    rdrift_Itr, _ = safe_spearman(df["drift"], df["I_tr"])
    return dict(
        n=n, L=L, seed=seed, n_eval=len(df),
        cv_gtr=cv(df["norm_gtr"].values), cv_gval=cv(df["norm_gval"].values),
        mean_norm_gtr=float(np.mean(df["norm_gtr"])),
        mean_norm_gval=float(np.mean(df["norm_gval"])),
        rho_A_Ival=rA_Ival, p_A_Ival=pA_Ival, rho_IP_Ival=rIP_Ival,
        identity_gap=(rIP_Ival - rA_Ival)
        if np.isfinite(rIP_Ival) and np.isfinite(rA_Ival) else np.nan,
        rho_A_G=rA_G, p_A_G=pA_G,
        rho_A_G_partial_Ival=rA_G_part,      # residual gap association
        rho_Ival_G=rIval_G,                  # shared-variable link (for context)
        rho_drift_Itr=rdrift_Itr)

## 6. Configuration
Edit `CFG` here. Set `quick=True` for a fast smoke test first; then set it to `False` and (optionally) add `12` to `n` for the full sweep. The full sweep is best run overnight; you can also run one `n` at a time and concatenate the CSVs.

In [ ]:
CFG = dict(
    n=[4, 6, 8, 10],        # add 12 if runtime allows
    L=[2, 3, 5],
    seeds=3,
    n_outer=200,            # Reptile outer steps for the meta-init
    n_eval=50,              # evaluation tasks per (n, L, seed)
    R=200,                  # random-init samples for the barren probe
    K=10,                   # inner adaptation steps
    inner_lr=0.5, outer_lr=0.3, init_scale=0.1,
    coupling_lo=0.5, coupling_hi=1.5, field_scale=0.5,
    base_seed=0,
    out_pertask="results_pertask.csv",
    out_summary="results_summary.csv",
    out_barren="results_barren.csv",
)

quick = True   # <-- set False for the full run
if quick:
    CFG.update(n=[4, 6], L=[2, 3], seeds=2, n_outer=30, n_eval=15, R=40)

CFG

## 7. Run the sweep

In [ ]:
def run(cfg):
    coeff_kw = dict(coupling_lo=cfg["coupling_lo"], coupling_hi=cfg["coupling_hi"],
                    field_scale=cfg["field_scale"])
    pertask_frames, summary_rows, barren_rows = [], [], []
    for n in cfg["n"]:
        dev = qml.device("default.qubit", wires=n)
        ops = build_ops(n)
        for L in cfg["L"]:
            for seed in range(cfg["seeds"]):
                t0 = time.time()
                rng = np.random.default_rng(cfg["base_seed"] + 1000*n + 100*L + seed)
                barren_rows.append(barren_row(dev, n, L, ops, rng, cfg["R"], coeff_kw, seed))
                theta0 = reptile_meta_init(dev, n, L, ops, rng, cfg["n_outer"], cfg["K"],
                                           cfg["inner_lr"], cfg["outer_lr"],
                                           cfg["init_scale"], coeff_kw)
                df = meta_eval(dev, n, L, ops, theta0, rng, cfg["n_eval"],
                               cfg["K"], cfg["inner_lr"], coeff_kw)
                df.insert(0, "seed", seed); df.insert(0, "L", L); df.insert(0, "n", n)
                pertask_frames.append(df)
                s = summarize(df, n, L, seed); summary_rows.append(s)
                bp = barren_rows[-1]
                print(f"[n={n} L={L} seed={seed}] params={2*n*L:3d} M={len(ops):3d} "
                      f"BPvar={bp['grad_comp_var']:.2e} |g|={s['mean_norm_gtr']:.3e} "
                      f"CV(gtr)={s['cv_gtr']:.3f} rho(A,Ival)={s['rho_A_Ival']:+.3f} "
                      f"rho(A,G)={s['rho_A_G']:+.3f} idgap={s['identity_gap']:+.3f} "
                      f"({time.time()-t0:.1f}s)", flush=True)
    pertask = pd.concat(pertask_frames, ignore_index=True)
    summary = pd.DataFrame(summary_rows)
    barren = pd.DataFrame(barren_rows)
    pertask.to_csv(cfg["out_pertask"], index=False)
    summary.to_csv(cfg["out_summary"], index=False)
    barren.to_csv(cfg["out_barren"], index=False)
    return pertask, summary, barren

t0 = time.time()
pertask, summary, barren = run(CFG)
print(f"\nTotal wall time: {time.time()-t0:.1f}s")
summary

## 8. Prediction-check summary (text)

In [ ]:
def print_report(summary, barren):
    agg = (summary.groupby(["n", "L"])
           .agg(cv_gtr=("cv_gtr", "mean"), mean_norm=("mean_norm_gtr", "mean"),
                rho_A_Ival=("rho_A_Ival", "mean"),
                identity_gap=("identity_gap", "mean")).reset_index())
    bp_agg = (barren.groupby(["n", "L"])
              .agg(BPvar=("grad_comp_var", "mean")).reset_index())
    agg = agg.merge(bp_agg, on=["n", "L"], how="left")

    print("[Concentration + identity dominance] expect as n,L up: "
          "BPvar down, mean|g| down, CV down, rho(A,Ival) high, id_gap -> 0")
    hdr = f"{'n':>3} {'L':>3} {'BPvar':>10} {'mean|g|':>10} {'CV(gtr)':>8} {'rho(A,Ival)':>12} {'id_gap':>8}"
    print(hdr); print("-"*len(hdr))
    for _, r in agg.sort_values(["L", "n"]).iterrows():
        print(f"{int(r['n']):>3} {int(r['L']):>3} {r['BPvar']:>10.2e} {r['mean_norm']:>10.3e} "
              f"{r['cv_gtr']:>8.3f} {r['rho_A_Ival']:>+12.3f} {r['identity_gap']:>+8.3f}")

    print("\n[Held-out gap NOT predicted] expect rho(A,G) weak & sign-UNSTABLE across seeds")
    piv = summary.pivot_table(index=["n", "L"], columns="seed", values="rho_A_G")
    hdr2 = f"{'n':>3} {'L':>3} " + " ".join(f"{'seed'+str(s):>9}" for s in piv.columns)
    print(hdr2); print("-"*len(hdr2))
    for (n, L), row in piv.iterrows():
        cells = " ".join(f"{row[s]:>+9.3f}" if np.isfinite(row[s]) else f"{'nan':>9}" for s in piv.columns)
        print(f"{int(n):>3} {int(L):>3} {cells}")

    print("\n[Control: is the gap association merely induced by I_val?] "
          "expect rho(A,G|I_val) close to 0 -> residual gap signal is negligible")
    ctrl = (summary.groupby(["n", "L"])
            .agg(raw=("rho_A_G", "mean"),
                 partial=("rho_A_G_partial_Ival", "mean"),
                 Ival_G=("rho_Ival_G", "mean")).reset_index())
    hdr3 = f"{'n':>3} {'L':>3} {'rho(A,G)':>10} {'rho(A,G|Ival)':>14} {'rho(Ival,G)':>12}"
    print(hdr3); print("-"*len(hdr3))
    for _, r in ctrl.sort_values(["L", "n"]).iterrows():
        print(f"{int(r['n']):>3} {int(r['L']):>3} {r['raw']:>+10.3f} "
              f"{r['partial']:>+14.3f} {r['Ival_G']:>+12.3f}")

print_report(summary, barren)

## 9. Prediction-check plots
Four panels visualise the prediction and the key control. With a full run these become manuscript figures: (a) gradient-norm CV vs $n$, (b) barren-plateau variance vs $n$, (c) the identity link $\rho(A,I_{\mathrm{val}})$ vs $n$, and (d) **the control** — raw $\rho(A,G)$ vs the $I_{\mathrm{val}}$-controlled partial $\rho(A,G\,|\,I_{\mathrm{val}})$ vs $n$. Panel (d) settles the yellow flag: if the raw gap association strengthens with $n$ but the partial one stays near zero, the gap signal was merely *induced* by the shared $I_{\mathrm{val}}$, consistent with the manuscript's reading.

In [ ]:
def plot_prediction(summary, barren):
    fig, axs = plt.subplots(2, 2, figsize=(11, 8))
    Ls = sorted(summary["L"].unique())

    # (a) gradient-norm CV vs n, per L  -> concentration
    for L in Ls:
        s = summary[summary["L"] == L].groupby("n")["cv_gtr"].mean()
        axs[0, 0].plot(s.index, s.values, "o-", label=f"L={L}")
    axs[0, 0].set_xlabel("qubits n"); axs[0, 0].set_ylabel(r"CV($\|g_{tr}\|$)")
    axs[0, 0].set_title("(a) gradient-norm concentration"); axs[0, 0].legend()

    # (b) barren-plateau variance vs n, per L (log) -> BP onset
    for L in Ls:
        b = barren[barren["L"] == L].groupby("n")["grad_comp_var"].mean()
        axs[0, 1].semilogy(b.index, b.values, "s-", label=f"L={L}")
    axs[0, 1].set_xlabel("qubits n"); axs[0, 1].set_ylabel(r"Var$[\partial L/\partial\theta_1]$")
    axs[0, 1].set_title("(b) barren-plateau probe"); axs[0, 1].legend()

    # (c) identity link rho(A,Ival) vs n, per L -> identity dominance
    for L in Ls:
        s = summary[summary["L"] == L].groupby("n")["rho_A_Ival"].mean()
        axs[1, 0].plot(s.index, s.values, "o-", label=f"L={L}")
    axs[1, 0].axhline(0, color="grey", lw=0.8)
    axs[1, 0].set_xlabel("qubits n"); axs[1, 0].set_ylabel(r"$\rho(A, I_{val})$")
    axs[1, 0].set_title("(c) identity link (local improvement)"); axs[1, 0].legend()

    # (d) THE CONTROL: raw rho(A,G) vs partial rho(A,G | Ival) vs n
    raw = summary.groupby("n")["rho_A_G"].mean()
    part = summary.groupby("n")["rho_A_G_partial_Ival"].mean()
    axs[1, 1].plot(raw.index, raw.values, "x--", color="crimson",
                   markersize=9, label=r"$\rho(A,G)$ raw")
    axs[1, 1].plot(part.index, part.values, "o-", color="navy",
                   label=r"$\rho(A,G\,|\,I_{val})$ partial")
    # per-seed scatter to show spread
    axs[1, 1].scatter(summary["n"], summary["rho_A_G"], c="crimson",
                      marker="x", alpha=0.35, zorder=1)
    axs[1, 1].scatter(summary["n"], summary["rho_A_G_partial_Ival"], c="navy",
                      marker="o", alpha=0.25, zorder=1)
    axs[1, 1].axhline(0, color="grey", lw=0.8)
    axs[1, 1].set_xlabel("qubits n"); axs[1, 1].set_ylabel("Spearman rho")
    axs[1, 1].set_title("(d) gap association: raw vs I_val-controlled")
    axs[1, 1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig("scaling_prediction.png", dpi=200, bbox_inches="tight")
    plt.show()

plot_prediction(summary, barren)

## 10. Reviewer-requested supplementary analyses
These address the referee directly: (10.1) the one-step identity vs the $K$-step experiment
($K{=}1$ where Eq. is exact vs $K{=}10$); (10.2) exponential fit of the barren-plateau variance;
(10.3) the i.i.d. dimensional null $1/\sqrt{2d}$; (10.4) multi-control partials
$\rho(A,G\,|\,I_\mathrm{val})$ and $\rho(A,G\,|\,I_\mathrm{val},L_\mathrm{val}(\theta_0))$
with hierarchical-bootstrap CIs; (10.5) meta-init vs random-init gradient statistics.

In [ ]:
from scipy.stats import rankdata

def rho(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3 or np.std(x[m]) == 0 or np.std(y[m]) == 0: return np.nan
    return spearmanr(x[m], y[m])[0]

def partial_multi(x, y, Z, df):
    d = df[[x, y] + Z].dropna()
    if len(d) < 5: return np.nan
    R = {c: rankdata(d[c].values) for c in [x, y] + Z}
    M = np.column_stack([np.ones(len(d))] + [R[c] for c in Z])
    res = lambda v: v - M @ np.linalg.lstsq(M, v, rcond=None)[0]
    ex, ey = res(R[x]), res(R[y])
    return np.corrcoef(ex, ey)[0, 1] if np.std(ex) and np.std(ey) else np.nan

def hier_ci(dfn, fn, B=1500, rng=np.random.default_rng(0)):
    cfgs = [g for _, g in dfn.groupby(["L", "seed"])]
    out = np.empty(B)
    for k in range(B):
        ch = [cfgs[i] for i in rng.integers(0, len(cfgs), len(cfgs))]
        fr = pd.concat([c.iloc[rng.integers(0, len(c), len(c))] for c in ch], ignore_index=True)
        out[k] = fn(fr)
    return tuple(np.nanpercentile(out, [2.5, 97.5]))

ns = sorted(pertask["n"].unique())
pertask["absG"] = pertask["G"].abs()

# 10.1 one-step (K=1) vs K-step identity link
print("10.1  identity link: rho(A, I_val) at K=1 (exact) vs K=%d" % CFG["K"])
print(f"  {'n':>3} {'rho(A,Ival_K1)':>15} {'rho(A,Ival_K)':>14}")
for n in ns:
    d = pertask[pertask.n == n]
    print(f"  {n:>3} {rho(d['align'], d['I_val_K1']):>+15.3f} {rho(d['align'], d['I_val']):>+14.3f}")

# 10.2 exponential fit of barren-plateau variance
print("\n10.2  Var[dL/dtheta1] ~ exp(-b n)  (per depth)")
for L in sorted(barren.L.unique()):
    s = barren[barren.L == L].groupby("n")["grad_comp_var"].mean()
    b = -np.polyfit(np.array(s.index, float), np.log(s.values), 1)[0]
    print(f"  L={L}: b = {b:.3f} per qubit")

# 10.3 i.i.d. dimensional null
print("\n10.3  CV(||g_tr||) observed vs null 1/sqrt(2d), d=2nL")
for n in ns:
    for L in sorted(pertask[pertask.n == n].L.unique()):
        v = pertask[(pertask.n == n) & (pertask.L == L)]["norm_gtr"].values
        cv = np.std(v)/np.mean(v); d = 2*n*L; null = 1/np.sqrt(2*d)
        print(f"  n={n} L={L}: obs={cv:.3f}  null={null:.3f}  ratio={cv/null:.2f}")

# 10.4 multi-control partials with hierarchical bootstrap CIs
print("\n10.4  gap association: raw / |I_val / |I_val,Lval0   (pooled per n, 95% CI)")
print(f"  {'n':>3} {'raw':>22} {'|I_val':>22} {'|I_val,Lval0':>22}")
for n in ns:
    d = pertask[pertask.n == n]
    r0 = rho(d["align"], d["G"]);  c0 = hier_ci(d, lambda f: rho(f["align"], f["G"]))
    r1 = partial_multi("align","G",["I_val"], d); c1 = hier_ci(d, lambda f: partial_multi("align","G",["I_val"], f))
    r2 = partial_multi("align","G",["I_val","Lval0"], d); c2 = hier_ci(d, lambda f: partial_multi("align","G",["I_val","Lval0"], f))
    print(f"  {n:>3} {r0:>+7.3f}[{c0[0]:+.2f},{c0[1]:+.2f}] {r1:>+7.3f}[{c1[0]:+.2f},{c1[1]:+.2f}] {r2:>+7.3f}[{c2[0]:+.2f},{c2[1]:+.2f}]")

# 10.5 meta-init vs random-init gradient component variance
print("\n10.5  gradient component variance: random-init (barren probe) vs meta-init (theta0)")
print(f"  {'n':>3} {'Var_random':>12} {'Var_metainit':>13}")
for n in ns:
    vr = barren[barren.n == n]["grad_comp_var"].mean()
    vm = pertask[pertask.n == n]["gtr1"].var()
    print(f"  {n:>3} {vr:>12.2e} {vm:>13.2e}")

These five outputs map one-to-one onto the referee's points: 10.1 addresses the one-step vs
$K$-step objection (the identity is strongest at $K{=}1$ and remains strong at $K{=}10$); 10.2/10.3
substantiate the barren-plateau/concentration claim (exponential decay; above the dimensional null);
10.4 is the decisive control (the gap association vanishes under the joint control); 10.5 connects
the random-init barren probe to the meta-init regime where the diagnostics are computed.

---
### Interpretation checklist
- **(a) & (b) trending down with $n$ (and $L$)** → gradient concentration / barren-plateau onset: the physical driver.
- **$\rho(A,I_{\mathrm{val}})$ high and rising, `identity_gap` $\to 0$** → the linearization identity increasingly governs the alignment diagnostic.
- **(d) raw $\rho(A,G)$ possibly nonzero, but partial $\rho(A,G\,|\,I_{\mathrm{val}})$ near zero across $n$** → any apparent gap association is *induced* by the shared $I_{\mathrm{val}}$, not a genuine held-out-gap signal. This is the control that closes the concern that alignment "becomes a gap predictor at scale."

If (a)/(b) trend down, (c) stays high, and (d)'s partial correlation stays near zero, the manuscript's prediction is **demonstrated across system sizes**, and Sec. IV C can be upgraded from *"we predict"* to *"we demonstrate."*

**Report back:** send `results_summary.csv`, `results_barren.csv`, `results_pertask.csv` (and `scaling_prediction.png`). These become new manuscript figures/tables, and the partial-correlation control becomes a sentence in Sec. IV C addressing the residual gap association directly.